<a href="https://colab.research.google.com/github/GregKyri/ToDrinQ_WP6/blob/main/conditioning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math as mh
import pandas as pd
import numpy as np

def conditioning(lanes,Partable2,dailyflow,Treatment):


    flowRate = max(dailyflow)/24  # m3/h
    #alk = Partable2['Total Alkalinity'][3]
    alk=20
    print("CONDITIONING")
    print("The total alkalinity (mg/l CaCO3) in the source water is",alk)
    # Select method
    if (Treatment['Treatment Technologies'].str.lower().str.strip() == "dosing of chemicals").any():
      condarea,condenergy =doschemi(flowRate,lanes,alk) #, units
    elif (Treatment['Treatment Technologies'].str.lower().str.strip() == "ph correction").any():
      condarea,condenergy =phcor(flowRate,lanes,alk) #, units
    elif (Treatment['Treatment Technologies'].str.lower().str.strip() == "remineralisation").any():
      condarea,condenergy =remin(flowRate,lanes,alk) #, units
    else:
      condarea,condenergy = 0,0

    return condarea,condenergy

def doschemi(flowRate,lanes,alk):
    # Sodium Carbonate (Na2CO3) Dosing Calculation
    """
    Calculates the Na2CO3 dosing requirements.

    Parameters:
    - alk: Initial alkalinity in mg/L as CaCO3
    - alkalinity_target: Target alkalinity in mg/L as CaCO3
    - daily_flow_m3: Daily water flow in cubic meters (m3/day)

    Returns:
    - Required Na2CO3 dosage (kg/day)
    - Pump flow rate (L/h)
    - Suggested storage tank capacity (m3)
    """
    # Required alkalinity increase
    alkalinity_target= float(input("Set target alkalinity in (mg/L CaCO3):"))
    delta_alkalinity = alkalinity_target - alk

    # Na2CO3 required (1 mg/L as CaCO3 = 1.06 mg/L Na2CO3)
    na2co3_concentration_mgL = delta_alkalinity * 1.06

    # Total Na2CO3 required per day (mg/h -> kg/h)
    na2co3_hourly_kg = (na2co3_concentration_mgL * flowRate) / 1000

    # Solution preparation: Assuming 10% Na2CO3 solution (w/w)
    solution_required_L_per_hour = na2co3_hourly_kg / 0.1

    # Dosing pump flow rate (L/h) per lane
    pump_flow_rate_Lph = solution_required_L_per_hour
    print(f"Dosing pump per lane flow rate : {pump_flow_rate_Lph:.2f} L/h")
    # Assume typical pump characteristics
    pump_head_bar = 3  #bar
    pump_efficiency = 0.75  #Pump efficiency
    pump_power_W = ((pump_flow_rate_Lph/(3600*1000)) * pump_head_bar*10.19 * 1000 * 9.81) / pump_efficiency
    pump_power_kW = (pump_power_W*1.2)/1000  # Typical small metering pump power

    # Energy
    condenergy=lanes*pump_power_kW*24 #KWh
    print(f"Dosing Pump power per lane: {pump_power_kW:.2f} kW")
    print(f"Conditioning energy consumpion per day: {condenergy:.2f} KWh")

    # Recommended storage tank capacity (30-day supply)
    print("We assume a 30-day storage area")
    storage_capacity_m3 = solution_required_L_per_hour*24*30*lanes / 1000
    print(f"Recommended storage tank capacity: {storage_capacity_m3:.2f} m3")
    storarea=mh.ceil(storage_capacity_m3/2)
    print(f"Required storage area: {storarea:.2f} m2")
    condarea=mh.ceil(storarea*1.1)
    print(f"Required conditioning area: {condarea:.2f} m2")
    return condarea,condenergy

def phcor(flowRate,lanes,alk):
    # Lime (Ca(OH)₂) Dosing Calculation
    """
    Calculates the Ca(OH)₂ dosing requirements.

    Parameters:
    - alk: Initial alkalinity in mg/L as CaCO3
    - alkalinity_target: Target alkalinity in mg/L as CaCO3
    - daily_flow_m3: Daily water flow in cubic meters (m3/day)

    Returns:
    - Required Ca(OH)₂ dosage (kg/30days)
    - Pump flow rate (L/h)
    - Suggested storage tank capacity (m3)
    - Suggested storage tank area (m2)
    """
    # Required alkalinity increase
    alkalinity_target= float(input("Set target alkalinity in (mg/L CaCO3):"))
    delta_alkalinity = alkalinity_target - alk

    # Convert to lime (Ca(OH)₂) requirement (mg/L as Ca(OH)₂)
    lime_dose_mg_per_L = delta_alkalinity * (74 / 100)  # Conversion factor

    # Total lime required per day (kg/hour)
    lime_required_kg_per_hour = (lime_dose_mg_per_L * flowRate) / 1000

    # Lime slurry concentration (g/L)
    lime_slurry_concentration = float(input("The Lime slurry concentration is (recommended is 5%):"))
    lime_slurry_concentration = lime_slurry_concentration*10 # Convert % to g/L
    slurry_concentration_kg_per_L = lime_required_kg_per_hour / (lime_slurry_concentration/1000)   # Convert % to kg/L*h

    # Required slurry volume per day (L/h)
    slurry_volume_L_per_hour = lime_required_kg_per_hour / slurry_concentration_kg_per_L

    # Storage requirement (m3)
    print("We assume a 30-day storage area")
    storage_capacity_m3 = slurry_volume_L_per_hour * 30 *lanes *24/1000
    print(f"Recommended storage tank capacity: {storage_capacity_m3:.2f} m3")
    storarea=mh.ceil(storage_capacity_m3/2)
    print(f"Required storage area: {storarea:.2f} m2")
    condarea=mh.ceil(storarea*1.1)
    print(f"Required conditioning area: {condarea:.2f} m2")

    # Pump flow rate (L/h) assuming continuous dosing
    pump_flow_rate_Lph = slurry_volume_L_per_hour
    print(f"Dosing pump per lane flow rate : {pump_flow_rate_Lph:.2f} L/h")
    # Assume typical pump characteristics
    pump_head_bar = 3  #bar
    pump_efficiency = 0.75  #Pump efficiency
    pump_power_W = ((pump_flow_rate_Lph/(3600*1000)) * pump_head_bar*10.19 * 1000 * 9.81) / pump_efficiency
    pump_power_kW = pump_power_W*1.2/1000  # Typical small metering pump power
    # Energy
    condenergy=mh.ceil(lanes*pump_power_kW*24) #KWh
    print(f"Dosing Pump power per lane: {pump_power_kW:.2f} kW")
    print(f"Conditioning energy consumpion per day: {condenergy:.2f} KWh")
    return condarea,condenergy

def remin(flowRate,lanes,alk):
    # CaCO₃ Dosing Calculation
    """
    Calculates the CaCO₃ dosing requirements.

    Parameters:
    - alk: Initial alkalinity in mg/L as CaCO3
    - alkalinity_target: Target alkalinity in mg/L as CaCO3
    - daily_flow_m3: Daily water flow in cubic meters (m3/day)

    Returns:
    - Required CaCO₃ dosage (kg/30days)
    - Pump flow rate (L/h)
    - Suggested storage tank capacity (m3)
    - Suggested storage tank area (m2)
    """
    # Required alkalinity increase
    alkalinity_target= float(input("Set target alkalinity in (mg/L CaCO3):"))
    delta_alkalinity = alkalinity_target - alk

    # Calculate required chemical dosage in kg/hour
    chemical_dosage_kg_hour = delta_alkalinity * flowRate/1000  # kg/hour

    # Calculate required volume of the chemical solution per day in liters
    chemical_concentration=float(input("Set the CaCO3 solution(recomended is 10%):"))
    volume_chemical_solution_L_hour = chemical_dosage_kg_hour * 1000 / (chemical_concentration*1000)  # L/hour

    # Storage requirement (m3)
    print("We assume a 30-day storage area")
    storage_capacity_m3 = volume_chemical_solution_L_hour * 30 *lanes *24/1000
    print(f"Recommended storage tank capacity: {storage_capacity_m3:.2f} m3")
    storarea=mh.ceil(storage_capacity_m3/2)
    print(f"Required storage area: {storarea:.2f} m2")
    condarea=mh.ceil(storarea*1.1)
    print(f"Required conditioning area: {condarea:.2f} m2")

    # Pump flow rate (L/h) assuming continuous dosing
    pump_flow_rate_Lph = volume_chemical_solution_L_hour
    print(f"Dosing pump per lane flow rate : {pump_flow_rate_Lph:.2f} L/h")
    # Assume typical pump characteristics
    pump_head_bar = 3  #bar
    pump_efficiency = 0.75  #Pump efficiency
    pump_power_W = ((pump_flow_rate_Lph/(3600*1000)) * pump_head_bar*10.19 * 1000 * 9.81) / pump_efficiency
    pump_power_kW = mh.ceil(pump_power_W*1.2/1000)  # Typical small metering pump power
    # Energy
    condenergy=mh.ceil(lanes*pump_power_kW*24) #KWh
    print(f"Dosing Pump power per lane: {pump_power_kW:.2f} kW")
    print(f"Conditioning energy consumpion per day: {condenergy:.2f} KWh")
    return condarea,condenergy